# 02 - Engenharia de Features das Cestas (Fase 2 · ML · Task 2)
**Squad 2 — Negócio: ML em Streaming | P2 · Recomendação de produtos no carrinho (Item2Vec)**
**Responsável:** Zaiden Emiliano Segundo Seleme · **Branch:** `feat/squad2-fase2-ml-zaiden`

---

### Objetivo
Preparar os dados do stream para o Item2Vec: transformar itens de pedido em **cestas codificadas**, separar **treino e teste por tempo** e montar o **hold-out** (remover 1 item da cesta e tentar recuperá-lo), o mesmo protocolo do Modelo A (FP-Growth, Lucas).

### Como o Item2Vec enxerga os dados
| Conceito do Word2Vec | No nosso problema |
|---|---|
| Frase | Uma cesta (`id_pedido`) |
| Palavra | Um SKU |
| Vocabulário | SKUs vistos pelo menos `MIN_COUNT` vezes no treino |
| Contexto (`window`) | Os outros SKUs da mesma cesta |
| Vetor da palavra (embedding) | Vetor denso de 16–32 números por SKU |

### Colunas existentes → entradas do modelo → por quê
| Tabela / coluna | Entra no modelo? | Por quê |
|---|---|---|
| `itens.id_pedido` | **Sim** (agrupa a cesta) | Define a "frase": SKUs do mesmo pedido são contexto um do outro. |
| `itens.sku` | **Sim** (é o token) | É o que o modelo aprende a representar e recomenda. Vira índice inteiro (label encoding). |
| `itens.dt_pedido` | Não (só no split) | Separa treino e teste por tempo. A EDA mostrou \|r\| ≤ 0,20 entre hora/dia da semana e a cesta: o contexto temporal não explica o que se compra junto. |
| `produtos.id_categoria` → `subcategoria`, `categoria_raiz` | Não (só no fallback) | O Item2Vec só usa SKUs. A categoria serve de plano B para SKU fora do vocabulário (cobre 99,9% do teste). |
| `itens.quantidade` | Não | Correlação de 0,93 com `n_skus`: o cliente leva quase sempre 1 unidade, então a quantidade não acrescenta informação sobre **o que** combina. |
| `itens.preco_unitario`, `desconto_aplicado`, `receita_*` | Não | O Word2Vec não recebe atributos numéricos; preço e desconto dizem quanto custa, não com o que se compra. |
| `itens.id_item_pedido` | Não | Identificador técnico, sem significado. |
| `bronze_*`, `silver_processed_at`, `ano`, `mes`, `flag_*` | Não | Metadados de linhagem e de qualidade da Fase 1. |

### Encoding, escala e redução de dimensionalidade (orientação do Tech Lead)
* **Encoding:** cada SKU vira um índice inteiro (`sku_idx`, 0 = mais vendido no treino). É o label encoding pedido para colunas categóricas.
* **Redução de dimensionalidade:** em one-hot, cada cesta seria um vetor com ~1,7 mil colunas, quase todas zero. O Item2Vec comprime cada SKU para **16–32 dimensões densas**: essa é a redução de dimensionalidade do modelo (seção 6 mede o ganho).
* **Escala:** não há colunas numéricas de magnitudes diferentes na entrada. A normalização acontece no modelo: a recomendação usa **similaridade de cosseno**, que divide cada vetor pelo seu tamanho (norma L2).
* **Colunas derivadas:** `n_skus` e `n_categorias_raiz` por cesta; `freq_treino` e `popularidade` (frequência escalada 0–1) por SKU, usadas no baseline e no desempate do fallback.

### Saídas (camada Gold da Fase 2)
| Tabela | Conteúdo | Usada por |
|---|---|---|
| `gold/item2vec_skus` | Um SKU por linha: `sku_idx`, frequência, popularidade, categoria, `no_vocabulario` | 03 (treino), baseline, fallback |
| `gold/item2vec_cestas_treino` | Uma cesta por linha com a lista de SKUs (as "frases") | 03 (treino) |
| `gold/item2vec_holdout` | Uma cesta de teste por linha: `carrinho` (entrada) e `item_removido` (resposta) | 03 (avaliação) e **Lucas** (mesmo teste) |

In [ ]:
%pip install python-dotenv==1.0.1 --quiet

In [ ]:
dbutils.library.restartPython()

## 0. Ambiente, credenciais, caminhos e parâmetros

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from pyspark.sql import functions as F, Window
from pyspark.sql.functions import col, lit

# Padronização de TimeZone Oficial da Squad 2 (Horário de Brasília - UTC-3)
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

dotenv_path = find_dotenv()
if not dotenv_path:
    candidatos = [
        os.path.join(os.getcwd(), ".env"),
        os.path.join(os.path.dirname(os.getcwd()), ".env"),
        os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), ".env"),
        "/Workspace/Shared/estagio_ed/.env",
        "/Workspace/Repos/estagio_ed/.env",
    ]
    dotenv_path = next((c for c in candidatos if os.path.isfile(c)), None)
load_dotenv(dotenv_path, override=True)

storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME")
client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")
assert storage_account and client_id and tenant_id and client_secret, "Credenciais do Azure ADLS não encontradas no .env"

fqdn = f"{storage_account}.dfs.core.windows.net"
adls_options = {
    f"fs.azure.account.auth.type.{fqdn}": "OAuth",
    f"fs.azure.account.oauth.provider.type.{fqdn}": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{fqdn}": client_id,
    f"fs.azure.account.oauth2.client.secret.{fqdn}": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{fqdn}": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token",
}

RAIZ_SQUAD = f"abfss://squad2@{fqdn}"
FONTES = {
    "itens":      f"{RAIZ_SQUAD}/grupo2/silver/ecommerce_itens_pedido",
    "produtos":   f"{RAIZ_SQUAD}/silver/ecommerce_produtos",
    "categorias": f"{RAIZ_SQUAD}/silver/ecommerce_categorias",
}
GOLD = f"{RAIZ_SQUAD}/fase2_ml/gold"
DESTINOS = {
    "skus":    f"{GOLD}/item2vec_skus",
    "treino":  f"{GOLD}/item2vec_cestas_treino",
    "holdout": f"{GOLD}/item2vec_holdout",
}

MIN_COUNT = 2          # EDA: mantém ~92% das vendas no vocabulário (com 5 seriam só 58%)
MIN_SKUS_CESTA = 2     # cesta com 1 SKU não tem contexto nem permite remover um item
PCT_TREINO = 0.8       # teste = últimos 20% das cestas válidas pelo instante do pedido

def ler_delta(caminho):
    return spark.read.format("delta").options(**adls_options).load(caminho)

def gravar_delta(df, destino):
    """Overwrite: as features são recalculadas inteiras a partir da Silver a cada execução."""
    (df.write.format("delta").options(**adls_options)
       .mode("overwrite").option("overwriteSchema", "true").save(destino))

def como_bigint(c):
    """Converte qualquer tipo (inclusive '1.0') para bigint; valor inválido vira nulo."""
    return F.expr(f"try_cast(try_cast(try_cast(`{c}` AS STRING) AS DOUBLE) AS BIGINT)")

## 1. Leitura da Silver: só as colunas que têm papel no modelo
Conforme a tabela de colunas acima: `id_pedido` e `sku` (entradas), `dt_pedido` (split) e a categoria do catálogo (fallback).

In [ ]:
cat = ler_delta(FONTES["categorias"]).select(
    como_bigint("id_categoria").alias("id_categoria"),
    F.trim("nome_categoria").alias("nome_categoria"),
    como_bigint("id_categoria_pai").alias("id_pai"))
nomes_raiz = cat.select(col("id_categoria").alias("id_raiz"), col("nome_categoria").alias("categoria_raiz"))
cat = (cat.withColumn("id_raiz", F.coalesce("id_pai", "id_categoria"))
          .join(nomes_raiz, "id_raiz", "left")
          .select("id_categoria", col("nome_categoria").alias("subcategoria"), "categoria_raiz"))
catalogo = (ler_delta(FONTES["produtos"])
            .select(F.trim("sku").alias("sku"), como_bigint("id_categoria").alias("id_categoria"))
            .dropDuplicates(["sku"])
            .join(cat, "id_categoria", "left")
            .drop("id_categoria"))

itens = (ler_delta(FONTES["itens"])
         .select("id_pedido", F.trim("sku").alias("sku"), "dt_pedido")
         .join(F.broadcast(catalogo), "sku", "left"))
print(f"Itens lidos: {itens.count():,}")

## 2. Regras de cesta
1. **Cada SKU conta uma vez por pedido.** A cesta é um conjunto: levar 3 unidades do mesmo produto não diz nada sobre o que combina com ele.
2. **Só cestas com 2+ SKUs.** Uma cesta de 1 SKU não tem "vizinho" para o Word2Vec e, no teste, ficaria vazia ao remover o item.

A cesta guarda os SKUs **ordenados**. Como a `window` cobre a cesta inteira, a ordem não muda o que o modelo aprende, mas a ordenação fixa deixa o resultado reprodutível.

In [ ]:
cestas = (itens.groupBy("id_pedido").agg(
              F.min("dt_pedido").alias("dt_pedido"),
              F.array_sort(F.collect_set("sku")).alias("skus"),
              F.countDistinct("categoria_raiz").alias("n_categorias_raiz"))
          .withColumn("n_skus", F.size("skus")))

total = cestas.count()
cestas = cestas.filter(col("n_skus") >= MIN_SKUS_CESTA)
print(f"Cestas: {total:,} | descartadas com < {MIN_SKUS_CESTA} SKUs: {total - cestas.count():,} | válidas: {cestas.count():,}")

## 3. Split temporal treino × teste
O teste fica com as **cestas mais recentes** (últimos 20% pelo instante do pedido). Assim o modelo é avaliado como seria usado: treinado no passado e recomendando no futuro. Um sorteio aleatório deixaria o modelo "ver o futuro" no treino e inflaria as métricas.

In [ ]:
# percentil sobre o epoch (segundos) e volta para timestamp
corte = cestas.agg(F.timestamp_seconds(
    F.percentile_approx(col("dt_pedido").cast("double"), PCT_TREINO, 10000))).first()[0]
cestas = cestas.withColumn("conjunto", F.when(col("dt_pedido") <= corte, "treino").otherwise("teste"))

print(f"Corte: {corte}")
(cestas.groupBy("conjunto").agg(F.count("*").alias("cestas"), F.min("dt_pedido").alias("de"),
                                F.max("dt_pedido").alias("ate"), F.round(F.avg("n_skus"), 2).alias("media_skus"))
       .orderBy(F.desc("conjunto")).show(truncate=False))

## 4. Vocabulário e encoding dos SKUs
A frequência é contada **só no treino**, para que nenhuma informação do teste vaze para o modelo. Cada SKU recebe:

* `freq_treino`: em quantas cestas de treino aparece;
* `popularidade`: `freq_treino` escalada entre 0 e 1 (min-max), para comparar SKUs independentemente do tamanho da base;
* `no_vocabulario`: `freq_treino ≥ MIN_COUNT`, ou seja, terá vetor no Item2Vec;
* `sku_idx`: índice inteiro (0 = mais vendido), só para quem está no vocabulário.

SKUs do catálogo que nunca apareceram no treino entram com frequência 0: o fallback por subcategoria pode recomendá-los só se não houver opção melhor.

In [ ]:
treino = cestas.filter(col("conjunto") == "treino")
freq = (treino.select(F.explode("skus").alias("sku")).groupBy("sku").agg(F.count("*").alias("freq_treino")))
fmin, fmax = freq.agg(F.min("freq_treino"), F.max("freq_treino")).first()

ordem = Window.orderBy(F.desc("freq_treino"), "sku")   # ~1,7 mil linhas: janela global sem custo relevante
skus = (catalogo.join(freq, "sku", "left").fillna(0, ["freq_treino"])
        .withColumn("popularidade", F.round((col("freq_treino") - fmin) / F.lit(fmax - fmin), 4))
        .withColumn("popularidade", F.greatest(col("popularidade"), F.lit(0.0)))
        .withColumn("no_vocabulario", col("freq_treino") >= MIN_COUNT))
vocab_idx = (skus.filter("no_vocabulario").select("sku", "freq_treino")
             .withColumn("sku_idx", F.row_number().over(ordem) - 1).drop("freq_treino"))
skus = skus.join(vocab_idx, "sku", "left")

gravar_delta(skus, DESTINOS["skus"])
skus = ler_delta(DESTINOS["skus"])

vendidos = skus.filter(col("freq_treino") > 0)
n_vocab, cobertura = vendidos.agg(
    F.sum(col("no_vocabulario").cast("int")),
    F.sum(F.when(col("no_vocabulario"), col("freq_treino"))) / F.sum("freq_treino")).first()
print(f"SKUs no catálogo: {skus.count():,} | vendidos no treino: {vendidos.count():,} | "
      f"no vocabulário (freq ≥ {MIN_COUNT}): {n_vocab:,} | ocorrências cobertas: {cobertura:.1%}")
skus.filter("no_vocabulario").orderBy("sku_idx").select("sku_idx", "sku", "freq_treino", "popularidade", "subcategoria").show(10, truncate=False)

## 5. Frases de treino e hold-out
**Treino:** cada cesta de treino vira uma frase com os SKUs do vocabulário. SKUs abaixo do `MIN_COUNT` saem da frase (o Word2Vec os ignoraria), e a cesta que ficar com menos de 2 SKUs sai do treino.

**Hold-out (leave-one-out):** em cada cesta de teste, 1 SKU é removido e vira a resposta; o restante é o `carrinho` que o modelo recebe. O item removido é escolhido por **hash de (`id_pedido`, `sku`)**, e não por sorteio: o resultado é o mesmo a cada execução e pode ser reaproveitado pelo Lucas, para que os dois modelos sejam medidos na mesma prova.

O item removido **não** é filtrado pelo vocabulário: se ele nunca apareceu no treino, o Item2Vec erra, e isso precisa aparecer na métrica.

In [ ]:
vocab = skus.filter("no_vocabulario").select("sku")

frases = (treino.select("id_pedido", "dt_pedido", "n_categorias_raiz", F.explode("skus").alias("sku"))
          .join(F.broadcast(vocab), "sku")
          .groupBy("id_pedido", "dt_pedido", "n_categorias_raiz")
          .agg(F.array_sort(F.collect_list("sku")).alias("skus"))
          .withColumn("n_skus", F.size("skus"))
          .filter(col("n_skus") >= MIN_SKUS_CESTA))
gravar_delta(frases, DESTINOS["treino"])

por_pedido = Window.partitionBy("id_pedido").orderBy(F.xxhash64("id_pedido", "sku"), "sku")
teste_itens = (cestas.filter(col("conjunto") == "teste")
               .select("id_pedido", "dt_pedido", F.explode("skus").alias("sku"))
               .withColumn("_ordem", F.row_number().over(por_pedido)))
removidos = teste_itens.filter("_ordem = 1").select("id_pedido", col("sku").alias("item_removido"))
carrinhos = (teste_itens.filter("_ordem > 1").groupBy("id_pedido", "dt_pedido")
             .agg(F.array_sort(F.collect_list("sku")).alias("carrinho")))
info_removido = skus.select(col("sku").alias("item_removido"),
                            col("no_vocabulario").alias("removido_no_vocabulario"),
                            col("subcategoria").alias("subcategoria_removido"))
holdout = (carrinhos.join(removidos, "id_pedido")
           .join(info_removido, "item_removido", "left")
           .fillna(False, ["removido_no_vocabulario"])
           .withColumn("tamanho_carrinho", F.size("carrinho")))
gravar_delta(holdout, DESTINOS["holdout"])

## 6. Conferência das saídas
Verifica que o teste não vazou para o treino, que nenhum carrinho ficou vazio e mostra o ganho da redução de dimensionalidade.

In [ ]:
frases, holdout, skus = (ler_delta(DESTINOS[k]) for k in ["treino", "holdout", "skus"])

vazamento = frases.join(holdout, "id_pedido").count()
data_treino_max = frases.agg(F.max("dt_pedido")).first()[0]
data_teste_min = holdout.agg(F.min("dt_pedido")).first()[0]
assert vazamento == 0, "Há pedidos do teste no treino"
assert holdout.filter(col("tamanho_carrinho") < 1).count() == 0, "Há carrinho vazio no hold-out"
assert data_treino_max <= data_teste_min, "O treino tem pedidos posteriores ao teste"
print(f"OK: sem vazamento | último pedido do treino {data_treino_max} ≤ primeiro do teste {data_teste_min}")

(frases.agg(F.count("*").alias("frases_treino"), F.sum("n_skus").alias("tokens"),
            F.round(F.avg("n_skus"), 2).alias("media_skus"), F.max("n_skus").alias("max_skus"))
       .show(truncate=False))
(holdout.agg(F.count("*").alias("cestas_teste"),
             F.round(F.avg("tamanho_carrinho"), 2).alias("media_carrinho"),
             F.round(100 * F.avg(col("removido_no_vocabulario").cast("int")), 1).alias("pct_removido_no_vocabulario"))
        .show(truncate=False))

n_vocab = skus.filter("no_vocabulario").count()
densidade = frases.agg(F.avg("n_skus")).first()[0] / n_vocab
print(f"One-hot: {n_vocab:,} colunas por cesta, {densidade:.2%} preenchidas (média de {densidade * n_vocab:.1f} SKUs).")
print(f"Item2Vec: 16 a 32 colunas densas por SKU, ou seja, {n_vocab // 32}x a {n_vocab // 16}x menos dimensões.")

## 7. Resultado
* **Entradas do modelo:** `id_pedido` (agrupa a cesta) e `sku` codificado em `sku_idx`. Nenhuma outra coluna entra no Item2Vec; o motivo de cada exclusão está na tabela do início.
* **Features geradas:** cestas como listas de SKUs (frases), vocabulário com `freq_treino`, `popularidade` (0–1) e `sku_idx`, além de `n_skus` e `n_categorias_raiz` por cesta.
* **Avaliação:** hold-out com as cestas mais recentes, 1 item removido por cesta de forma reprodutível. **Combinar com o Lucas** o uso de `gold/item2vec_holdout`, para que o FP-Growth seja medido no mesmo teste.
* **Próximo passo:** `03_treino_item2vec`. Treinar o Word2Vec (gensim) nas frases, recomendar por cosseno com a média dos vetores do carrinho, aplicar o fallback por subcategoria e comparar com o baseline top-5 mais vendidos (Hit@5, Precision@5 e cobertura do catálogo).